# MeetScribe Batch - Trascrizione multipla con GPU

Trascrivi **tutti** i file audio in una cartella in un colpo solo.

**Come usare:**
1. `Runtime > Change runtime type > T4 GPU`
2. Esegui le celle in ordine
3. Carica i file audio nella cartella `/content/trascrizioni`
4. Lancia la pipeline — i modelli vengono caricati **una sola volta**
5. A fine elaborazione scarica lo zip con tutte le trascrizioni

## 1. Setup: installa MeetScribe + dipendenze GPU

In [ ]:
import os

REPO_URL = 'https://github.com/paratusapp/meet-scribe.git'

repo_dir = '/content/meet-scribe'
if not os.path.exists(repo_dir):
    !git clone {REPO_URL} {repo_dir}
else:
    !cd {repo_dir} && git pull

%cd {repo_dir}
!pip install -e . -q

# Verifica GPU
import torch
print(f"\nGPU disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Token HuggingFace

Serve per i modelli pyannote. Prendi il token da: https://huggingface.co/settings/tokens

In [ ]:
import os
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    print('Token caricato da Colab Secrets')
except Exception:
    token = 'hf_YOUR_TOKEN_HERE'  # <-- sostituisci
    print('Token impostato manualmente')

os.environ['HUGGING_FACE_TOKEN'] = token
os.environ['HF_TOKEN'] = token

## 3. Carica i file audio

Carica i file nella cartella `/content/trascrizioni`. Puoi:
- **Opzione A**: usare il file browser di Colab (pannello sinistro) e trascinare i file
- **Opzione B**: eseguire la cella sotto per fare upload

In [ ]:
import os
from pathlib import Path
from google.colab import files

INPUT_DIR = Path('/content/trascrizioni')
INPUT_DIR.mkdir(exist_ok=True)

print(f'Cartella input: {INPUT_DIR}')
print('Seleziona i file audio da trascrivere...\n')

uploaded = files.upload()

for filename, content in uploaded.items():
    dest = INPUT_DIR / filename
    dest.write_bytes(content)
    print(f'  -> {dest.name} ({len(content) / 1e6:.1f} MB)')

print(f'\nTotale: {len(uploaded)} file caricati in {INPUT_DIR}')

## 4. Esegui la pipeline batch

Carica i modelli una sola volta, poi trascrive tutti i file in sequenza.
I risultati vanno in `/content/trascrizioni_testuali/`.

In [ ]:
import time
import tempfile
from pathlib import Path
import yaml

from meet_scribe.audio_extractor import extract_audio, get_audio_duration
from meet_scribe.diarizer import diarize
from meet_scribe.transcriber import load_whisper_model, transcribe
from meet_scribe.formatter import (
    format_timestamp,
    merge_diarization_and_transcription,
    save_txt,
)

# --- Configurazione ---
INPUT_DIR = Path('/content/trascrizioni')
OUTPUT_DIR = Path('/content/trascrizioni_testuali')
OUTPUT_DIR.mkdir(exist_ok=True)

# Carica config dal repo
config_path = Path('/content/meet-scribe/config.yaml')
with open(config_path, encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Estensioni audio supportate
AUDIO_EXTS = {'.m4a', '.mp4', '.wav', '.mp3', '.ogg', '.flac', '.webm', '.aac', '.wma'}

# Trova tutti i file audio
audio_files = sorted([
    f for f in INPUT_DIR.iterdir()
    if f.suffix.lower() in AUDIO_EXTS
])

if not audio_files:
    print(f'Nessun file audio trovato in {INPUT_DIR}')
    print(f'Estensioni supportate: {", ".join(AUDIO_EXTS)}')
else:
    print(f'Trovati {len(audio_files)} file audio:')
    for f in audio_files:
        print(f'  - {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

    # --- Carica modelli UNA SOLA VOLTA ---
    print(f'\n{"="*60}')
    print(f'  Caricamento modelli (una tantum)...')
    print(f'{"="*60}')

    w_config = config['whisper']
    model_name = w_config.get('model', 'large-v3-turbo')
    compute = w_config.get('compute_type', 'int8')
    print(f'  Whisper: {model_name}')
    whisper_model = load_whisper_model(model_size=model_name, compute_type=compute)
    print(f'  Whisper caricato.')
    print(f'  Pyannote: si carica al primo file.\n')

    # Parametri dal config
    diar_config = config.get('diarization', {})
    raw_hp = diar_config.get('hyperparams') or {}
    hyperparams = {k: v for k, v in raw_hp.items() if v is not None}
    lang = w_config.get('language')
    raw_vad = w_config.get('vad') or {}
    vad_params = {k: v for k, v in raw_vad.items() if v is not None}

    # --- Pipeline batch ---
    total_start = time.time()
    results = []

    for idx, audio_path in enumerate(audio_files, 1):
        file_start = time.time()
        print(f'\n{"="*60}')
        print(f'  [{idx}/{len(audio_files)}] {audio_path.name}')
        print(f'{"="*60}')

        try:
            with tempfile.TemporaryDirectory() as tmp_dir:
                # 1. Estrazione audio
                print('  [1/4] Estrazione audio...')
                wav_path = extract_audio(
                    audio_path, Path(tmp_dir),
                    sample_rate=config['audio']['sample_rate'],
                )
                duration = get_audio_duration(wav_path)
                print(f'         Durata: {format_timestamp(duration)}')

                # 2. Diarization
                print('  [2/4] Speaker diarization...')
                diarization_segments = diarize(
                    wav_path,
                    min_speakers=diar_config.get('min_speakers'),
                    max_speakers=diar_config.get('max_speakers'),
                    hyperparams=hyperparams if hyperparams else None,
                )
                n_speakers = len(set(s['speaker'] for s in diarization_segments))
                print(f'         {n_speakers} speaker, {len(diarization_segments)} segmenti')

                # 3. Trascrizione (riusa il modello gia caricato)
                print('  [3/4] Trascrizione...')
                transcription_segments, words, detected_lang = transcribe(
                    wav_path, model=whisper_model,
                    language=lang,
                    beam_size=w_config.get('beam_size', 5),
                    initial_prompt=w_config.get('initial_prompt'),
                    vad_params=vad_params if vad_params else None,
                )
                print(f'         Lingua: {detected_lang}, {len(words)} parole')

            # 4. Merge e salvataggio
            print('  [4/4] Merge e salvataggio...')
            merged = merge_diarization_and_transcription(
                diarization_segments, transcription_segments, words=words
            )

            speakers = sorted(set(s['speaker'] for s in merged))
            output_data = {
                'file': audio_path.name,
                'lingua': detected_lang,
                'durata': format_timestamp(duration),
                'num_speaker': len(speakers),
                'speaker': speakers,
                'trascrizione': merged,
            }

            # Salva .txt con lo stesso nome del file audio
            txt_path = OUTPUT_DIR / f'{audio_path.stem}.txt'
            save_txt(output_data, txt_path)

            elapsed = time.time() - file_start
            print(f'  OK: {txt_path.name} ({len(merged)} segmenti, {elapsed:.0f}s)')
            results.append((audio_path.name, 'OK', elapsed))

        except Exception as e:
            elapsed = time.time() - file_start
            print(f'  ERRORE: {e}')
            results.append((audio_path.name, f'ERRORE: {e}', elapsed))

    # --- Riepilogo ---
    total_elapsed = time.time() - total_start
    print(f'\n{"="*60}')
    print(f'  RIEPILOGO BATCH')
    print(f'{"="*60}')
    for name, status, elapsed in results:
        print(f'  {name}: {status} ({elapsed:.0f}s)')
    print(f'\n  Tempo totale: {total_elapsed:.0f}s')
    print(f'  Output in: {OUTPUT_DIR}/')

## 5. Anteprima risultati

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path('/content/trascrizioni_testuali')

txt_files = sorted(OUTPUT_DIR.glob('*.txt'))
print(f'Trascrizioni generate: {len(txt_files)}\n')

for txt_file in txt_files:
    content = txt_file.read_text(encoding='utf-8')
    lines = content.split('\n')
    print(f'--- {txt_file.name} ({len(lines)} righe) ---')
    # Mostra prime 20 righe
    print('\n'.join(lines[:20]))
    if len(lines) > 20:
        print(f'  ... ({len(lines) - 20} righe rimanenti)')
    print()

## 6. Scarica ZIP con tutte le trascrizioni

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

OUTPUT_DIR = Path('/content/trascrizioni_testuali')
ZIP_PATH = Path('/content/trascrizioni_testuali.zip')

txt_files = sorted(OUTPUT_DIR.glob('*.txt'))

if not txt_files:
    print('Nessuna trascrizione da zippare.')
else:
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for txt_file in txt_files:
            zf.write(txt_file, txt_file.name)
            print(f'  Aggiunto: {txt_file.name}')

    size_mb = ZIP_PATH.stat().st_size / 1e6
    print(f'\nZIP creato: {ZIP_PATH.name} ({size_mb:.2f} MB, {len(txt_files)} file)')
    print('Download in corso...')
    files.download(str(ZIP_PATH))